# MMRL / BayesMMRL 动态 R 权重融合验证（repo-forward 修正版）

这个 notebook 用于验证两版动态融合：

1. **无 β 版**：基于 `u / margin / JS` 的规则式权重；
2. **有 β 版**：在验证集上搜索 `β_u, β_m, β_d`。

## 关键修正

本版不再手写裸 `model.forward()` / `model.forward_eval()` 来重建分支，而是严格走项目标准评估路径：

```python
outputs = trainer.method.forward_eval(batch, eval_ctx)
z_c = outputs.logits
z_r = outputs.aux_logits["rep"]
z_repo_fusion = outputs.aux_logits["fusion"]
z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)
```

这样可以保证：

- MMRL 使用 `BaseMMRLFamilyMethod.forward_eval()` 的真实分支；
- BayesMMRL 使用 `BayesMMRLMethod.forward_eval()` 的真实 MC / aggregation / posterior 配置；
- repo 原始评估基线来自 `select_eval_logits()`，和 `trainer.test()` 一致；
- 动态融合在 **logits 空间** 做：

```python
z_dynamic = (1 - omega_r) * z_c + omega_r * z_r
```

## 输出

运行最后一个单元后，只生成一个主要分析文件：

```text
output_refactor/analysis/dynamic_r_fusion_repo_forward_corrected/dynamic_r_fusion_repo_forward_corrected_report.xlsx
```

Excel 内含多个 sheet：

- `README`
- `Conclusions`
- `ConfigAudit`
- `Accuracy`
- `SanityChecks`
- `Transitions`
- `BranchOracle`
- `Quadrants`
- `BetaTop`
- `ChangedSamples`
- `FullDiagnostics`

如果你的实验不是 `default` tag，而是 sweep / exp_config 覆盖过配置，请在 `CASES` 中显式填写 `exp_config` 或 `opts`。

In [1]:
from pathlib import Path

# ====== 改这里 ======
REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATA_ROOT = REPO_ROOT / "DATASETS"

CASES = [
    {
        "name": "MMRL_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/MMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1",
        # 可选：如果当初用了额外 exp config，在这里填
        # "exp_config": "configs/experiments/xxx.yaml",
        # 可选：如果当初命令行有额外 opts，在这里填列表
        # "opts": ["MMRL.ALPHA", "0.7"],
    },
    {
        "name": "BayesMMRL_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1",
        # "exp_config": "",
        # "opts": [],
    },
]

TEST_SPLIT = "test"
BETA_SELECT_SPLIT = "val"

# 调试时可设成小整数；正式实验设 None。
MAX_BATCHES = None

# β 网格
BETA_GRID_VALUES = [0.0, 0.5, 1.0, 2.0, 4.0]

# 固定 MC eval 随机种子，降低 BayesMMRL 重复运行波动。
EVAL_SEED = 2026

# 修正分支提取后，第一次运行建议 False，避免读旧缓存。
SAVE_NPZ_CACHE = True
USE_NPZ_CACHE_IF_EXISTS = False

OUT_DIR = REPO_ROOT / "output_refactor/analysis/dynamic_r_fusion_repo_forward_corrected"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.exists(), f"REPO_ROOT 不存在: {REPO_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT 不存在: {DATA_ROOT}"
for case in CASES:
    assert Path(case["case_root"]).exists(), f'case_root 不存在: {case["case_root"]}'

print("REPO_ROOT =", REPO_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUT_DIR   =", OUT_DIR)
print("CASES     =", len(CASES))

REPO_ROOT = /root/autodl-tmp/MMRL
DATA_ROOT = /root/autodl-tmp/MMRL/DATASETS
OUT_DIR   = /root/autodl-tmp/MMRL/output_refactor/analysis/dynamic_r_fusion_repo_forward_corrected
CASES     = 2


In [2]:
import os
import sys
import math
import json
import random
import importlib
from types import SimpleNamespace
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from core.config import setup_cfg
from dassl.engine import build_trainer
from core.utils import import_optional_modules

# 与 run.py 对齐：注册 datasets + trainer。
import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])
importlib.import_module("trainers.refactor_runner")

print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())

torch = 2.11.0+cu128
cuda available = True


In [3]:
METHOD_CFG_MAP = {
    "MMRL": "configs/methods/mmrl.yaml",
    "MMRLMix": "configs/methods/mmrl_mix.yaml",
    "MMRLpp": "configs/methods/mmrlpp.yaml",
    "MMRLPP": "configs/methods/mmrlpp.yaml",
    "BayesMMRL": "configs/methods/bayesmmrl.yaml",
    "ClipAdapters": "configs/methods/clip_adapters.yaml",
    "ClipADAPTER": "configs/methods/clip_adapters.yaml",
}

PROTOCOL_CFG_MAP = {
    "B2N": "configs/protocols/b2n.yaml",
    "FS": "configs/protocols/fs.yaml",
    "CD": "configs/protocols/cd.yaml",
}

PROTOCOL_TO_SUBSAMPLE = {
    "B2N": "base",
    "FS": "all",
    "CD": "all",
}

def decode_backbone_from_dir(token: str) -> str:
    # output_refactor 中通常把 ViT-B/16 写成 ViT-B-16。
    if token.startswith("ViT-") and token.count("-") >= 2:
        head, patch = token.rsplit("-", 1)
        return f"{head}/{patch}"
    return token

def infer_case_metadata(case_root: Path):
    case_root = Path(case_root).expanduser().resolve()
    parts = case_root.parts
    if "output_refactor" not in parts:
        raise ValueError(f"case_root 中找不到 output_refactor: {case_root}")

    idx = parts.index("output_refactor")
    method = parts[idx + 1]
    protocol = parts[idx + 2]
    phase = parts[idx + 3]
    dataset = parts[idx + 4]
    shots_str = parts[idx + 5]
    backbone_token = parts[idx + 6]
    tag = parts[idx + 7]
    seed_dir = parts[idx + 8]

    if not shots_str.startswith("shots_"):
        raise ValueError(f"无法从路径解析 shots: {shots_str}")
    if not seed_dir.startswith("seed"):
        raise ValueError(f"无法从路径解析 seed: {seed_dir}")

    return {
        "method": method,
        "protocol": protocol,
        "phase": phase,
        "dataset": dataset,
        "shots": int(shots_str.split("_", 1)[1]),
        "backbone": decode_backbone_from_dir(backbone_token),
        "tag": tag,
        "seed": int(seed_dir.replace("seed", "")),
        "case_root": case_root,
    }

def make_args_from_case(case):
    case_root = Path(case["case_root"]).expanduser().resolve()
    meta = infer_case_metadata(case_root)

    method = meta["method"]
    protocol = meta["protocol"]
    dataset = meta["dataset"]

    if method not in METHOD_CFG_MAP:
        raise KeyError(f"METHOD_CFG_MAP 缺少方法: {method}")
    if protocol not in PROTOCOL_CFG_MAP:
        raise KeyError(f"PROTOCOL_CFG_MAP 缺少协议: {protocol}")

    extra_opts = list(case.get("opts", []) or [])
    base_opts = [
        "DATASET.NUM_SHOTS", str(meta["shots"]),
        "DATASET.SUBSAMPLE_CLASSES", str(PROTOCOL_TO_SUBSAMPLE[protocol]),
        "MODEL.BACKBONE.NAME", str(meta["backbone"]),
    ]

    args = SimpleNamespace(
        root=str(DATA_ROOT),
        output_dir=str(case_root),
        dataset_config_file=f"configs/datasets/{dataset}.yaml",
        method_config_file=str(case.get("method_config_file", METHOD_CFG_MAP[method])),
        protocol_config_file=str(case.get("protocol_config_file", PROTOCOL_CFG_MAP[protocol])),
        runtime_config_file=str(case.get("runtime_config_file", "configs/runtime/default.yaml")),
        exp_config=str(case.get("exp_config", "") or ""),
        method=method,
        protocol=protocol,
        exec_mode=str(case.get("exec_mode", "online")),
        seed=int(meta["seed"]),
        trainer="RefactorRunner",
        eval_only=True,
        model_dir=str(case_root),
        load_epoch=case.get("load_epoch", None),
        no_train=True,
        opts=base_opts + extra_opts,
    )
    return args, meta

def build_trainer_for_case(case):
    args, meta = make_args_from_case(case)
    cfg = setup_cfg(args)
    trainer = build_trainer(cfg)
    trainer.load_model(str(meta["case_root"]), epoch=args.load_epoch)
    trainer.set_model_mode("eval")
    return trainer, meta, args

def get_cfg_alpha(trainer):
    cfg = trainer.cfg
    if str(cfg.METHOD.NAME) == "BayesMMRL":
        return float(cfg.BAYES_MMRL.ALPHA)
    if hasattr(cfg, "MMRL") and hasattr(cfg.MMRL, "ALPHA"):
        return float(cfg.MMRL.ALPHA)
    if hasattr(trainer.method, "method_cfg") and hasattr(trainer.method.method_cfg, "ALPHA"):
        return float(trainer.method.method_cfg.ALPHA)
    return 0.7

def audit_trainer_config(trainer, meta, args, case_name):
    cfg = trainer.cfg
    rows = []

    def add(key, value, expected=None):
        rows.append({
            "case_name": case_name,
            "key": key,
            "value": str(value),
            "expected_or_inferred": "" if expected is None else str(expected),
            "match": "" if expected is None else bool(str(value) == str(expected)),
        })

    add("cfg.METHOD.NAME", cfg.METHOD.NAME, meta["method"])
    add("cfg.METHOD.EXEC_MODE", cfg.METHOD.EXEC_MODE, args.exec_mode)
    add("cfg.PROTOCOL.NAME", cfg.PROTOCOL.NAME, meta["protocol"])
    add("cfg.PROTOCOL.PHASE", getattr(cfg.PROTOCOL, "PHASE", None), meta["phase"])
    add("cfg.DATASET.NAME", cfg.DATASET.NAME, meta["dataset"])
    add("cfg.DATASET.NUM_SHOTS", cfg.DATASET.NUM_SHOTS, meta["shots"])
    add("cfg.DATASET.SUBSAMPLE_CLASSES", cfg.DATASET.SUBSAMPLE_CLASSES, PROTOCOL_TO_SUBSAMPLE[meta["protocol"]])
    add("cfg.MODEL.BACKBONE.NAME", cfg.MODEL.BACKBONE.NAME, meta["backbone"])
    add("cfg.SEED", cfg.SEED, meta["seed"])
    add("cfg.OUTPUT_DIR", cfg.OUTPUT_DIR, meta["case_root"])

    if str(cfg.METHOD.NAME) == "BayesMMRL":
        add("cfg.BAYES_MMRL.ALPHA", cfg.BAYES_MMRL.ALPHA, 0.7)
        add("cfg.BAYES_MMRL.N_MC_TEST", cfg.BAYES_MMRL.N_MC_TEST)
        add("cfg.BAYES_MMRL.USE_MEAN_MAIN_MC_REP", cfg.BAYES_MMRL.USE_MEAN_MAIN_MC_REP)
        add("cfg.BAYES_MMRL.EVAL_MODE", cfg.BAYES_MMRL.EVAL_MODE)
        add("cfg.BAYES_MMRL.EVAL_USE_POSTERIOR_MEAN", cfg.BAYES_MMRL.EVAL_USE_POSTERIOR_MEAN)
        add("cfg.BAYES_MMRL.EVAL_AGGREGATION", cfg.BAYES_MMRL.EVAL_AGGREGATION)
        add("cfg.BAYES_MMRL.REP_SIGMA_MODE", cfg.BAYES_MMRL.REP_SIGMA_MODE)
        add("cfg.BAYES_MMRL.REP_PRIOR_STD", cfg.BAYES_MMRL.REP_PRIOR_STD)
        add("cfg.BAYES_MMRL.REP_KL_WEIGHT", cfg.BAYES_MMRL.REP_KL_WEIGHT)
    else:
        add("cfg.MMRL.ALPHA", getattr(cfg.MMRL, "ALPHA", None), 0.7)
        add("cfg.MMRL.REG_WEIGHT", getattr(cfg.MMRL, "REG_WEIGHT", None))

    add("args.dataset_config_file", args.dataset_config_file)
    add("args.method_config_file", args.method_config_file)
    add("args.protocol_config_file", args.protocol_config_file)
    add("args.runtime_config_file", args.runtime_config_file)
    add("args.exp_config", args.exp_config)
    add("args.opts", args.opts)

    return pd.DataFrame(rows)

In [4]:
def set_eval_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def collect_repo_forward_logits(trainer, split="test", max_batches=None, seed=2026):
    """
    严格使用 repo 标准 eval path:
      method.forward_eval -> method.select_eval_logits

    返回:
      z_c          = outputs.logits
      z_r          = outputs.aux_logits["rep"]
      z_aux_fusion = outputs.aux_logits["fusion"]
      z_repo_eval  = method.select_eval_logits(outputs, eval_ctx)
    """
    set_eval_seed(seed)
    trainer.set_model_mode("eval")

    if split == "val" and getattr(trainer, "val_loader", None) is not None:
        data_loader = trainer.val_loader
        actual_split = "val"
    else:
        data_loader = trainer.test_loader
        actual_split = "test"

    eval_ctx = trainer.executor.build_eval_context(trainer, actual_split)

    z_c_all, z_r_all, z_aux_fusion_all, z_repo_eval_all, y_all, index_all = [], [], [], [], [], []
    n_seen = 0

    for batch_idx, batch in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        image = batch["img"].to(trainer.device)
        label = batch["label"].to(trainer.device).long()

        outputs = trainer.method.forward_eval(
            {"img": image, "label": label},
            eval_ctx,
        )

        if outputs.aux_logits is None or "rep" not in outputs.aux_logits or "fusion" not in outputs.aux_logits:
            raise RuntimeError(
                "outputs.aux_logits 必须包含 'rep' 和 'fusion'。"
                f" 当前 aux_logits keys={None if outputs.aux_logits is None else list(outputs.aux_logits.keys())}"
            )

        z_c = outputs.logits
        z_r = outputs.aux_logits["rep"]
        z_aux_fusion = outputs.aux_logits["fusion"]
        z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)

        if not (z_c.shape == z_r.shape == z_aux_fusion.shape == z_repo_eval.shape):
            raise RuntimeError(
                f"logits shape 不一致: z_c={tuple(z_c.shape)}, z_r={tuple(z_r.shape)}, "
                f"z_aux_fusion={tuple(z_aux_fusion.shape)}, z_repo_eval={tuple(z_repo_eval.shape)}"
            )

        bsz = int(label.shape[0])
        z_c_all.append(z_c.detach().float().cpu())
        z_r_all.append(z_r.detach().float().cpu())
        z_aux_fusion_all.append(z_aux_fusion.detach().float().cpu())
        z_repo_eval_all.append(z_repo_eval.detach().float().cpu())
        y_all.append(label.detach().cpu())
        index_all.append(torch.arange(n_seen, n_seen + bsz, dtype=torch.long))
        n_seen += bsz

    if not y_all:
        raise RuntimeError(f"No batches collected for split={split}")

    return {
        "actual_split": actual_split,
        "z_c": torch.cat(z_c_all, dim=0).numpy(),
        "z_r": torch.cat(z_r_all, dim=0).numpy(),
        "z_aux_fusion": torch.cat(z_aux_fusion_all, dim=0).numpy(),
        "z_repo_eval": torch.cat(z_repo_eval_all, dim=0).numpy(),
        "y": torch.cat(y_all, dim=0).numpy(),
        "index": torch.cat(index_all, dim=0).numpy(),
    }

def cache_path_for(case_name, split):
    safe = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in case_name)
    return OUT_DIR / f"{safe}_{split}_repo_forward_logits.npz"

def load_or_collect_case_split(trainer, case_name, split, max_batches=None, seed=2026):
    path = cache_path_for(case_name, split)
    if USE_NPZ_CACHE_IF_EXISTS and path.exists():
        data = dict(np.load(path, allow_pickle=True))
        data["actual_split"] = str(data["actual_split"].item() if hasattr(data["actual_split"], "item") else data["actual_split"])
        return data

    data = collect_repo_forward_logits(
        trainer=trainer,
        split=split,
        max_batches=max_batches,
        seed=seed,
    )
    if SAVE_NPZ_CACHE:
        np.savez_compressed(path, **data)
    return data

In [5]:
EPS = 1e-12

def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    m = np.max(z, axis=1, keepdims=True)
    logsum = m + np.log(np.sum(np.exp(z - m), axis=1, keepdims=True))
    return z - logsum

def top1_acc_from_logits(z, y):
    return float(np.mean(np.argmax(z, axis=1) == y))

def n_correct_from_logits(z, y):
    return int(np.sum(np.argmax(z, axis=1) == y))

def nll_from_logits(z, y):
    logp = log_softmax_np(z)
    return float(-np.mean(logp[np.arange(len(y)), y]))

def brier_from_logits(z, y):
    p = softmax_np(z)
    onehot = np.zeros_like(p)
    onehot[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - onehot) ** 2, axis=1)))

def ece_from_logits(z, y, n_bins=10):
    p = softmax_np(z)
    conf = np.max(p, axis=1)
    pred = np.argmax(p, axis=1)
    correct = (pred == y).astype(float)
    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)
        if np.any(mask):
            ece += (np.mean(mask)) * abs(float(np.mean(conf[mask])) - float(np.mean(correct[mask])))
    return float(ece)

def metrics_from_logits(name, z, y):
    return {
        "variant": name,
        "accuracy": top1_acc_from_logits(z, y),
        "n_correct": n_correct_from_logits(z, y),
        "n": int(len(y)),
        "nll": nll_from_logits(z, y),
        "brier": brier_from_logits(z, y),
        "ece_10bin": ece_from_logits(z, y, n_bins=10),
    }

def entropy_norm(p):
    p = np.clip(p, EPS, 1.0)
    h = -np.sum(p * np.log(p), axis=1)
    return h / np.log(p.shape[1])

def margin_prob(p):
    top2 = np.partition(p, kth=-2, axis=1)[:, -2:]
    top2 = np.sort(top2, axis=1)
    return top2[:, 1] - top2[:, 0]

def js_div_norm(p, q):
    p = np.clip(p, EPS, 1.0)
    q = np.clip(q, EPS, 1.0)
    m = 0.5 * (p + q)
    js = 0.5 * np.sum(p * (np.log(p) - np.log(m)), axis=1) + 0.5 * np.sum(q * (np.log(q) - np.log(m)), axis=1)
    return js / np.log(2.0)

def compute_signals(z_c, z_r):
    p_c = softmax_np(z_c)
    p_r = softmax_np(z_r)
    u_c = entropy_norm(p_c)
    u_r = entropy_norm(p_r)
    m_c = margin_prob(p_c)
    m_r = margin_prob(p_r)
    d_cr = js_div_norm(p_c, p_r)
    a_cr = 1.0 - d_cr
    return {"p_c": p_c, "p_r": p_r, "u_c": u_c, "u_r": u_r, "m_c": m_c, "m_r": m_r, "d_cr": d_cr, "a_cr": a_cr}

def omega_no_beta(signals, prior_c=0.7, prior_r=0.3):
    r_r = np.clip(1.0 - signals["u_r"], 0.0, 1.0) * np.clip(signals["m_r"], 0.0, 1.0) * np.clip(signals["a_cr"], 0.0, 1.0)
    r_c = np.clip(1.0 - signals["u_c"], 0.0, 1.0) * np.clip(signals["m_c"], 0.0, 1.0)
    den = prior_c * r_c + prior_r * r_r + EPS
    w = (prior_r * r_r) / den
    return np.clip(w, 0.0, 1.0), r_r, r_c

def omega_beta(signals, beta_u, beta_m, beta_d, prior_c=0.7, prior_r=0.3):
    raw = (
        beta_u * (signals["u_c"] - signals["u_r"])
        + beta_m * (signals["m_r"] - signals["m_c"])
        - beta_d * signals["d_cr"]
    )
    raw = np.clip(raw, -30.0, 30.0)
    s = np.exp(raw)
    w = (prior_r * s) / (prior_c + prior_r * s + EPS)
    return np.clip(w, 0.0, 1.0), s

def fuse_logits(z_c, z_r, omega):
    return (1.0 - omega[:, None]) * z_c + omega[:, None] * z_r

def summarize_omega(prefix, omega, prior_r):
    return {
        f"{prefix}_omega_mean": float(np.mean(omega)),
        f"{prefix}_omega_median": float(np.median(omega)),
        f"{prefix}_omega_min": float(np.min(omega)),
        f"{prefix}_omega_max": float(np.max(omega)),
        f"{prefix}_omega_gt_prior_r_count": int(np.sum(omega > prior_r)),
        f"{prefix}_omega_gt_prior_r_rate": float(np.mean(omega > prior_r)),
    }

def pick_best_beta(z_c, z_r, y, signals, prior_c=0.7, prior_r=0.3, grid_values=None):
    if grid_values is None:
        grid_values = [0.0, 0.5, 1.0, 2.0, 4.0]
    rows = []
    for bu, bm, bd in product(grid_values, grid_values, grid_values):
        w, _ = omega_beta(signals, bu, bm, bd, prior_c=prior_c, prior_r=prior_r)
        z = fuse_logits(z_c, z_r, w)
        row = metrics_from_logits("beta_candidate", z, y)
        row.update({
            "beta_u": float(bu),
            "beta_m": float(bm),
            "beta_d": float(bd),
            "beta_sum": float(bu + bm + bd),
            "omega_mean": float(np.mean(w)),
            "omega_median": float(np.median(w)),
            "omega_gt_prior_r_rate": float(np.mean(w > prior_r)),
        })
        rows.append(row)
    grid_df = pd.DataFrame(rows)
    grid_df = grid_df.sort_values(
        ["accuracy", "beta_sum", "beta_u", "beta_m", "beta_d"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)
    return grid_df.iloc[0].to_dict(), grid_df

def true_label_rank(z, y):
    order = np.argsort(-z, axis=1)
    ranks = np.empty(len(y), dtype=np.int64)
    for i, yi in enumerate(y):
        ranks[i] = int(np.where(order[i] == yi)[0][0]) + 1
    return ranks

def transition_report(case_name, y, z_base, z_variant, variant_name):
    pred_base = np.argmax(z_base, axis=1)
    pred_var = np.argmax(z_variant, axis=1)
    base_correct = pred_base == y
    var_correct = pred_var == y
    changed = pred_base != pred_var
    return {
        "case_name": case_name,
        "variant": variant_name,
        "n": int(len(y)),
        "base_acc": float(np.mean(base_correct)),
        "variant_acc": float(np.mean(var_correct)),
        "delta_acc": float(np.mean(var_correct) - np.mean(base_correct)),
        "changed_pred_count": int(np.sum(changed)),
        "changed_pred_rate": float(np.mean(changed)),
        "fix_base_error_count": int(np.sum((~base_correct) & var_correct)),
        "break_base_correct_count": int(np.sum(base_correct & (~var_correct))),
        "net_fix_minus_break": int(np.sum((~base_correct) & var_correct) - np.sum(base_correct & (~var_correct))),
    }

def branch_oracle_report(case_name, y, z_c, z_r, z_repo_eval):
    pc = np.argmax(z_c, axis=1)
    pr = np.argmax(z_r, axis=1)
    pe = np.argmax(z_repo_eval, axis=1)
    c_ok = pc == y
    r_ok = pr == y
    e_ok = pe == y
    return {
        "case_name": case_name,
        "n": int(len(y)),
        "C_acc": float(np.mean(c_ok)),
        "R_acc": float(np.mean(r_ok)),
        "repo_eval_acc": float(np.mean(e_ok)),
        "oracle_choose_C_or_R_acc": float(np.mean(c_ok | r_ok)),
        "C_wrong_R_correct_count": int(np.sum((~c_ok) & r_ok)),
        "C_correct_R_wrong_count": int(np.sum(c_ok & (~r_ok))),
        "both_correct_count": int(np.sum(c_ok & r_ok)),
        "both_wrong_count": int(np.sum((~c_ok) & (~r_ok))),
        "C_R_argmax_agree_rate": float(np.mean(pc == pr)),
    }

In [6]:
def build_diagnostics(case_name, split, data, alpha, prior_c, prior_r, beta_params=None):
    z_c = data["z_c"]
    z_r = data["z_r"]
    z_aux_fusion = data["z_aux_fusion"]
    z_repo_eval = data["z_repo_eval"]
    y = data["y"].astype(int)
    index = data["index"].astype(int)

    signals = compute_signals(z_c, z_r)
    w_nb, rr_nb, rc_nb = omega_no_beta(signals, prior_c=prior_c, prior_r=prior_r)
    z_no_beta = fuse_logits(z_c, z_r, w_nb)

    if beta_params is None:
        beta_params = {"beta_u": 0.0, "beta_m": 0.0, "beta_d": 0.0}

    w_b, _ = omega_beta(
        signals,
        beta_params["beta_u"],
        beta_params["beta_m"],
        beta_params["beta_d"],
        prior_c=prior_c,
        prior_r=prior_r,
    )
    z_beta = fuse_logits(z_c, z_r, w_b)

    z_formula = alpha * z_c + (1.0 - alpha) * z_r
    p_prob_orig = prior_c * signals["p_c"] + prior_r * signals["p_r"]

    pred_c = np.argmax(z_c, axis=1)
    pred_r = np.argmax(z_r, axis=1)
    pred_aux_fusion = np.argmax(z_aux_fusion, axis=1)
    pred_repo_eval = np.argmax(z_repo_eval, axis=1)
    pred_formula = np.argmax(z_formula, axis=1)
    pred_prob_orig = np.argmax(p_prob_orig, axis=1)
    pred_no_beta = np.argmax(z_no_beta, axis=1)
    pred_beta = np.argmax(z_beta, axis=1)

    diag = pd.DataFrame({
        "case_name": case_name,
        "split": split,
        "index": index,
        "y": y,

        "pred_C": pred_c,
        "pred_R": pred_r,
        "pred_aux_fusion": pred_aux_fusion,
        "pred_repo_eval": pred_repo_eval,
        "pred_formula_alpha": pred_formula,
        "pred_control_prob_prior": pred_prob_orig,
        "pred_no_beta_logit": pred_no_beta,
        "pred_beta_tuned_logit": pred_beta,

        "correct_C": (pred_c == y).astype(int),
        "correct_R": (pred_r == y).astype(int),
        "correct_aux_fusion": (pred_aux_fusion == y).astype(int),
        "correct_repo_eval": (pred_repo_eval == y).astype(int),
        "correct_formula_alpha": (pred_formula == y).astype(int),
        "correct_control_prob_prior": (pred_prob_orig == y).astype(int),
        "correct_no_beta_logit": (pred_no_beta == y).astype(int),
        "correct_beta_tuned_logit": (pred_beta == y).astype(int),

        "rank_true_C": true_label_rank(z_c, y),
        "rank_true_R": true_label_rank(z_r, y),
        "rank_true_repo_eval": true_label_rank(z_repo_eval, y),

        "u_C": signals["u_c"],
        "u_R": signals["u_r"],
        "margin_C": signals["m_c"],
        "margin_R": signals["m_r"],
        "js_norm_CR": signals["d_cr"],
        "agree_norm_CR": signals["a_cr"],

        "r_R_no_beta": rr_nb,
        "r_C_no_beta": rc_nb,
        "omega_no_beta": w_nb,
        "omega_beta_tuned": w_b,

        "C_R_argmax_agree": (pred_c == pred_r).astype(int),

        "repo_to_no_beta_changed": (pred_repo_eval != pred_no_beta).astype(int),
        "repo_to_beta_changed": (pred_repo_eval != pred_beta).astype(int),
        "no_beta_fix_repo_error": ((pred_repo_eval != y) & (pred_no_beta == y)).astype(int),
        "no_beta_break_repo_correct": ((pred_repo_eval == y) & (pred_no_beta != y)).astype(int),
        "beta_fix_repo_error": ((pred_repo_eval != y) & (pred_beta == y)).astype(int),
        "beta_break_repo_correct": ((pred_repo_eval == y) & (pred_beta != y)).astype(int),
    })

    tensors = {
        "signals": signals,
        "z_formula_alpha": z_formula,
        "p_prob_orig": p_prob_orig,
        "z_no_beta": z_no_beta,
        "z_beta": z_beta,
        "omega_no_beta": w_nb,
        "omega_beta": w_b,
    }
    return diag, tensors

def accuracy_table_for_case(case_name, split, data, alpha, prior_c, prior_r, tensors):
    y = data["y"].astype(int)
    rows = []
    variants = [
        ("C_only_outputs.logits", data["z_c"]),
        ("R_only_aux_logits_rep", data["z_r"]),
        ("repo_aux_fusion", data["z_aux_fusion"]),
        ("repo_eval_select_eval_logits", data["z_repo_eval"]),
        (f"formula_logit_alpha_{alpha:.3f}", tensors["z_formula_alpha"]),
        ("no_beta_logit_dynamic", tensors["z_no_beta"]),
        ("beta_tuned_logit_dynamic", tensors["z_beta"]),
    ]

    for name, z in variants:
        row = metrics_from_logits(name, z, y)
        row.update({"case_name": case_name, "split": split})
        rows.append(row)

    p_prob = tensors["p_prob_orig"]
    pred = np.argmax(p_prob, axis=1)
    onehot = np.zeros_like(p_prob)
    onehot[np.arange(len(y)), y] = 1.0
    rows.append({
        "case_name": case_name,
        "split": split,
        "variant": f"control_prob_prior_{prior_c:.3f}C_{prior_r:.3f}R",
        "accuracy": float(np.mean(pred == y)),
        "n_correct": int(np.sum(pred == y)),
        "n": int(len(y)),
        "nll": float(-np.mean(np.log(np.clip(p_prob[np.arange(len(y)), y], EPS, 1.0)))),
        "brier": float(np.mean(np.sum((p_prob - onehot) ** 2, axis=1))),
        "ece_10bin": np.nan,
    })
    return pd.DataFrame(rows)

def sanity_checks_for_case(case_name, split, data, alpha, prior_c, prior_r, tensors):
    z_c = data["z_c"]
    z_r = data["z_r"]
    z_aux_fusion = data["z_aux_fusion"]
    z_repo_eval = data["z_repo_eval"]
    z_formula = alpha * z_c + (1.0 - alpha) * z_r

    pred_aux = np.argmax(z_aux_fusion, axis=1)
    pred_repo = np.argmax(z_repo_eval, axis=1)

    row = {
        "case_name": case_name,
        "split": split,
        "n": int(len(data["y"])),
        "num_classes": int(z_c.shape[1]),
        "cfg_alpha": float(alpha),
        "prior_c_used": float(prior_c),
        "prior_r_used": float(prior_r),
        "max_abs_formula_alpha_vs_aux_fusion": float(np.max(np.abs(z_formula - z_aux_fusion))),
        "formula_alpha_matches_aux_fusion_atol_1e-4": bool(np.max(np.abs(z_formula - z_aux_fusion)) < 1e-4),
        "max_abs_repo_eval_vs_aux_fusion": float(np.max(np.abs(z_repo_eval - z_aux_fusion))),
        "repo_eval_same_argmax_as_aux_fusion_rate": float(np.mean(pred_aux == pred_repo)),
        **summarize_omega("no_beta", tensors["omega_no_beta"], prior_r),
        **summarize_omega("beta", tensors["omega_beta"], prior_r),
    }
    return pd.DataFrame([row])

def quadrant_table(case_name, diag):
    d = diag.copy()
    conditions = [
        (d["correct_C"] == 1) & (d["correct_R"] == 1),
        (d["correct_C"] == 1) & (d["correct_R"] == 0),
        (d["correct_C"] == 0) & (d["correct_R"] == 1),
        (d["correct_C"] == 0) & (d["correct_R"] == 0),
    ]
    labels = ["C_correct_R_correct", "C_correct_R_wrong", "C_wrong_R_correct", "C_wrong_R_wrong"]
    d["quadrant"] = np.select(conditions, labels, default="unknown")

    agg_cols = [
        "u_C", "u_R", "margin_C", "margin_R", "js_norm_CR",
        "r_R_no_beta", "r_C_no_beta", "omega_no_beta", "omega_beta_tuned",
        "correct_repo_eval", "correct_no_beta_logit", "correct_beta_tuned_logit",
        "repo_to_no_beta_changed", "repo_to_beta_changed",
        "no_beta_fix_repo_error", "no_beta_break_repo_correct",
        "beta_fix_repo_error", "beta_break_repo_correct",
    ]

    rows = []
    for q, g in d.groupby("quadrant", dropna=False):
        row = {
            "case_name": case_name,
            "quadrant": q,
            "count": int(len(g)),
            "rate": float(len(g) / len(d)),
        }
        for c in agg_cols:
            row[f"mean_{c}"] = float(g[c].mean()) if len(g) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def changed_samples_table(case_name, diag, top_k=200):
    d = diag.copy()
    mask = (d["repo_to_no_beta_changed"] == 1) | (d["repo_to_beta_changed"] == 1)
    cols = [
        "case_name", "split", "index", "y",
        "pred_C", "pred_R", "pred_repo_eval", "pred_no_beta_logit", "pred_beta_tuned_logit",
        "correct_C", "correct_R", "correct_repo_eval", "correct_no_beta_logit", "correct_beta_tuned_logit",
        "omega_no_beta", "omega_beta_tuned", "u_C", "u_R", "margin_C", "margin_R", "js_norm_CR",
        "rank_true_C", "rank_true_R", "rank_true_repo_eval",
        "no_beta_fix_repo_error", "no_beta_break_repo_correct",
        "beta_fix_repo_error", "beta_break_repo_correct",
    ]
    out = d.loc[mask, cols].copy()
    out["importance"] = (
        out["no_beta_fix_repo_error"] + out["no_beta_break_repo_correct"]
        + out["beta_fix_repo_error"] + out["beta_break_repo_correct"]
    )
    return out.sort_values(["importance", "omega_no_beta", "omega_beta_tuned"], ascending=[False, False, False]).head(top_k).reset_index(drop=True)

In [7]:
def analyze_one_case(case):
    case_name = case["name"]
    print(f"\n===== {case_name} =====")
    trainer, meta, args = build_trainer_for_case(case)

    alpha = get_cfg_alpha(trainer)
    prior_c = alpha
    prior_r = 1.0 - alpha
    audit_df = audit_trainer_config(trainer, meta, args, case_name)

    # β 选择集：优先 val，没有 val_loader 时 fallback 到 test，并在 SanityChecks 里标记。
    val_data = load_or_collect_case_split(
        trainer,
        case_name=case_name,
        split=BETA_SELECT_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    val_split_actual = val_data["actual_split"]
    val_signals = compute_signals(val_data["z_c"], val_data["z_r"])

    best_beta, beta_grid_df = pick_best_beta(
        val_data["z_c"],
        val_data["z_r"],
        val_data["y"].astype(int),
        val_signals,
        prior_c=prior_c,
        prior_r=prior_r,
        grid_values=BETA_GRID_VALUES,
    )
    beta_params = {
        "beta_u": float(best_beta["beta_u"]),
        "beta_m": float(best_beta["beta_m"]),
        "beta_d": float(best_beta["beta_d"]),
    }

    print("beta_select_split requested/actual:", BETA_SELECT_SPLIT, "/", val_split_actual)
    print("best beta:", beta_params, "select_acc=", best_beta["accuracy"])

    # 测试集最终报告。
    test_data = load_or_collect_case_split(
        trainer,
        case_name=case_name,
        split=TEST_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    test_split_actual = test_data["actual_split"]

    diag, tensors = build_diagnostics(
        case_name=case_name,
        split=test_split_actual,
        data=test_data,
        alpha=alpha,
        prior_c=prior_c,
        prior_r=prior_r,
        beta_params=beta_params,
    )

    acc_df = accuracy_table_for_case(case_name, test_split_actual, test_data, alpha, prior_c, prior_r, tensors)
    sanity_df = sanity_checks_for_case(case_name, test_split_actual, test_data, alpha, prior_c, prior_r, tensors)
    sanity_df["beta_select_split_requested"] = BETA_SELECT_SPLIT
    sanity_df["beta_select_split_actual"] = val_split_actual
    sanity_df["beta_selection_uses_test_leakage"] = bool(val_split_actual == "test" and BETA_SELECT_SPLIT == "val")
    sanity_df["beta_u"] = beta_params["beta_u"]
    sanity_df["beta_m"] = beta_params["beta_m"]
    sanity_df["beta_d"] = beta_params["beta_d"]

    transitions_df = pd.DataFrame([
        transition_report(
            case_name,
            test_data["y"].astype(int),
            test_data["z_repo_eval"],
            tensors["z_no_beta"],
            "no_beta_logit_dynamic_vs_repo_eval",
        ),
        transition_report(
            case_name,
            test_data["y"].astype(int),
            test_data["z_repo_eval"],
            tensors["z_beta"],
            "beta_tuned_logit_dynamic_vs_repo_eval",
        ),
    ])

    oracle_df = pd.DataFrame([branch_oracle_report(
        case_name,
        test_data["y"].astype(int),
        test_data["z_c"],
        test_data["z_r"],
        test_data["z_repo_eval"],
    )])

    quad_df = quadrant_table(case_name, diag)
    changed_df = changed_samples_table(case_name, diag)

    beta_top_df = beta_grid_df.head(30).copy()
    beta_top_df.insert(0, "case_name", case_name)
    beta_top_df.insert(1, "selection_split_actual", val_split_actual)
    beta_top_df.insert(2, "selection_uses_test_leakage", bool(val_split_actual == "test" and BETA_SELECT_SPLIT == "val"))

    print(acc_df[["variant", "accuracy", "n_correct", "nll", "brier", "ece_10bin"]])
    print(sanity_df.T)

    return {
        "audit": audit_df,
        "accuracy": acc_df,
        "sanity": sanity_df,
        "transitions": transitions_df,
        "oracle": oracle_df,
        "quadrants": quad_df,
        "beta_top": beta_top_df,
        "changed": changed_df,
        "diagnostics": diag,
    }

In [8]:
def build_readme_df():
    return pd.DataFrame([
        {"section": "Purpose", "content": "验证 MMRL/BayesMMRL 训练后 logits 空间动态 R 权重融合；不重新训练模型。"},
        {"section": "Correct repo path", "content": "outputs = trainer.method.forward_eval(batch, eval_ctx); repo baseline = trainer.method.select_eval_logits(outputs, eval_ctx)。"},
        {"section": "Branch definitions", "content": "C = outputs.logits; R = outputs.aux_logits['rep']; aux fusion = outputs.aux_logits['fusion']; repo eval = select_eval_logits。"},
        {"section": "Dynamic fusion", "content": "z_dynamic = (1-omega_r)*z_c + omega_r*z_r；可靠性信号由 softmax(z_c), softmax(z_r) 计算。"},
        {"section": "No-beta omega", "content": "omega = prior_R*r_R/(prior_C*r_C + prior_R*r_R + eps), r_R=(1-u_R)*margin_R*(1-JS/log2), r_C=(1-u_C)*margin_C。"},
        {"section": "Beta omega", "content": "omega = prior_R*exp(beta_u*(u_C-u_R)+beta_m*(margin_R-margin_C)-beta_d*JS_norm)/(prior_C + prior_R*exp(...))。"},
        {"section": "Key sanity", "content": "SanityChecks 中检查 formula_alpha 与 aux_fusion 是否一致；BayesMMRL 正式 baseline 使用 repo_eval_select_eval_logits。"},
        {"section": "Leakage warning", "content": "如果 beta_select_split_actual=test，说明没有 val_loader，β 在测试集选择，不能作为正式无泄漏结果。"},
    ])

def add_conclusion_rows(accuracy_df, sanity_df, transitions_df, oracle_df):
    rows = []
    for case_name, g in accuracy_df.groupby("case_name"):
        acc = {r["variant"]: r for _, r in g.iterrows()}
        repo = acc.get("repo_eval_select_eval_logits", {})
        nb = acc.get("no_beta_logit_dynamic", {})
        bt = acc.get("beta_tuned_logit_dynamic", {})
        c = acc.get("C_only_outputs.logits", {})
        r = acc.get("R_only_aux_logits_rep", {})
        s = sanity_df[sanity_df["case_name"] == case_name].iloc[0].to_dict()
        oracle = oracle_df[oracle_df["case_name"] == case_name].iloc[0].to_dict()

        nb_delta = float(nb.get("accuracy", np.nan) - repo.get("accuracy", np.nan))
        bt_delta = float(bt.get("accuracy", np.nan) - repo.get("accuracy", np.nan))
        best_variant = g.sort_values("accuracy", ascending=False).iloc[0]["variant"]

        reasons = []
        if s.get("beta_selection_uses_test_leakage", False):
            reasons.append("β selection fell back to test: leakage risk.")
        if abs(float(s.get("max_abs_repo_eval_vs_aux_fusion", 0.0))) > 1e-4:
            reasons.append("repo_eval differs from aux_fusion; official baseline is repo_eval.")
        if nb_delta <= 0 and bt_delta <= 0:
            reasons.append("dynamic fusion does not beat repo eval on this split.")
        else:
            reasons.append("dynamic fusion beats repo eval; inspect Transitions for fix-vs-break source.")

        rows.append({
            "case_name": case_name,
            "repo_eval_acc": repo.get("accuracy", np.nan),
            "C_acc": c.get("accuracy", np.nan),
            "R_acc": r.get("accuracy", np.nan),
            "no_beta_acc": nb.get("accuracy", np.nan),
            "beta_tuned_acc": bt.get("accuracy", np.nan),
            "no_beta_minus_repo": nb_delta,
            "beta_tuned_minus_repo": bt_delta,
            "best_variant": best_variant,
            "oracle_choose_C_or_R_acc": oracle.get("oracle_choose_C_or_R_acc", np.nan),
            "C_wrong_R_correct_count": oracle.get("C_wrong_R_correct_count", np.nan),
            "C_correct_R_wrong_count": oracle.get("C_correct_R_wrong_count", np.nan),
            "beta_u": s.get("beta_u", np.nan),
            "beta_m": s.get("beta_m", np.nan),
            "beta_d": s.get("beta_d", np.nan),
            "no_beta_omega_mean": s.get("no_beta_omega_mean", np.nan),
            "beta_omega_mean": s.get("beta_omega_mean", np.nan),
            "diagnosis": " ".join(reasons),
        })
    return pd.DataFrame(rows)

def write_report(results, report_path):
    audit_df = pd.concat([r["audit"] for r in results], ignore_index=True)
    accuracy_df = pd.concat([r["accuracy"] for r in results], ignore_index=True)
    sanity_df = pd.concat([r["sanity"] for r in results], ignore_index=True)
    transitions_df = pd.concat([r["transitions"] for r in results], ignore_index=True)
    oracle_df = pd.concat([r["oracle"] for r in results], ignore_index=True)
    quadrants_df = pd.concat([r["quadrants"] for r in results], ignore_index=True)
    beta_top_df = pd.concat([r["beta_top"] for r in results], ignore_index=True)
    changed_df = pd.concat([r["changed"] for r in results], ignore_index=True)
    diagnostics_df = pd.concat([r["diagnostics"] for r in results], ignore_index=True)
    conclusions_df = add_conclusion_rows(accuracy_df, sanity_df, transitions_df, oracle_df)

    sheets = {
        "README": build_readme_df(),
        "Conclusions": conclusions_df,
        "ConfigAudit": audit_df,
        "Accuracy": accuracy_df,
        "SanityChecks": sanity_df,
        "Transitions": transitions_df,
        "BranchOracle": oracle_df,
        "Quadrants": quadrants_df,
        "BetaTop": beta_top_df,
        "ChangedSamples": changed_df,
        "FullDiagnostics": diagnostics_df,
    }

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        for name, df in sheets.items():
            df.to_excel(writer, sheet_name=name, index=False)

    return sheets

In [9]:
# ====== 运行主分析 ======

all_results = []
for case in CASES:
    result = analyze_one_case(case)
    all_results.append(result)

REPORT_PATH = OUT_DIR / "dynamic_r_fusion_repo_forward_corrected_report.xlsx"
sheets = write_report(all_results, REPORT_PATH)

print("\nSaved report to:")
print(REPORT_PATH)

print("\nConclusions:")
display(sheets["Conclusions"])

print("\nAccuracy:")
display(sheets["Accuracy"])


===== MMRL_16shot_seed1 =====
Loading trainer: RefactorRunner
Loading dataset: Caltech101
Reading split from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_zhou_Caltech101.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ----------
Dataset    Caltech101
# classes  100
# train_x  1,600
# val      400
# test     2,465
---------  ----------


[MMRL] trainable params: {'representation_learner.compound_rep_tokens_r2tproj.1.bias', 'representation_learner.compound_rep_tokens_r2tproj.5.bias', 'representation_learner.compound_rep_tokens_r2vproj.6.weight', 'representation_learner.compound_rep_tokens_r2tproj.2.weight', 'representation_learner.compound_rep_tokens_r2tproj.3.bias', 'representation_learner.compound_rep_tokens_r2vproj.5.weight', 'representation_learner.compound_rep_tokens_r2tproj.5.weight', 'representation_learner.compound_rep_tokens_r2tproj.4.bias', 'representation_learner.compound_rep_tokens_r2vproj.0.weight', 'representation_learner.compound_rep_tokens_r2vproj.2.weight', 'representation_learner.compound_rep_tokens_r2vproj.4.bias', 'representation_learner.compound_rep_tokens_r2vproj.1.bias', 'representation_learner.compound_rep_tokens_r2tproj.4.weight', 'representation_learner.compound_rep_tokens_r2vproj.1.weight', 'representation_learner.compound_rep_tokens_r2tproj.6.weight', 'representation_learner.compound_rep_toke

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


beta_select_split requested/actual: val / val
best beta: {'beta_u': 0.0, 'beta_m': 0.0, 'beta_d': 2.0} select_acc= 0.9525
                            variant  accuracy  n_correct       nll     brier  \
0             C_only_outputs.logits  0.965923       2381  0.106166  0.051074   
1             R_only_aux_logits_rep  0.008519         21  7.533094  1.237190   
2                   repo_aux_fusion  0.953753       2351  0.177742  0.074454   
3      repo_eval_select_eval_logits  0.953753       2351  0.177742  0.074454   
4         formula_logit_alpha_0.700  0.953753       2351  0.177742  0.074454   
5             no_beta_logit_dynamic  0.964300       2377  0.109472  0.053431   
6          beta_tuned_logit_dynamic  0.966734       2383  0.110020  0.052877   
7  control_prob_prior_0.700C_0.300R  0.965517       2380  0.457563  0.158586   

   ece_10bin  
0   0.003124  
1   0.400570  
2   0.047139  
3   0.047139  
4   0.047139  
5   0.003513  
6   0.006402  
7        NaN  
                      

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


beta_select_split requested/actual: val / val
best beta: {'beta_u': 1.0, 'beta_m': 0.0, 'beta_d': 0.5} select_acc= 0.9325
                            variant  accuracy  n_correct       nll     brier  \
0             C_only_outputs.logits  0.956998       2359  0.114737  0.058786   
1             R_only_aux_logits_rep  0.001217          3  7.155358  1.288794   
2                   repo_aux_fusion  0.954970       2354  0.186937  0.078790   
3      repo_eval_select_eval_logits  0.954970       2354  0.186937  0.078790   
4         formula_logit_alpha_0.700  0.954970       2354  0.186937  0.078790   
5             no_beta_logit_dynamic  0.955375       2355  0.120027  0.059703   
6          beta_tuned_logit_dynamic  0.956187       2357  0.134235  0.063337   
7  control_prob_prior_0.700C_0.300R  0.956187       2357  0.467310  0.173228   

   ece_10bin  
0   0.014613  
1   0.460993  
2   0.065103  
3   0.065103  
4   0.065103  
5   0.014430  
6   0.026027  
7        NaN  
                      

,case_name,repo_eval_acc,C_acc,R_acc,no_beta_acc,beta_tuned_acc,no_beta_minus_repo,beta_tuned_minus_repo,best_variant,oracle_choose_C_or_R_acc,C_wrong_R_correct_count,C_correct_R_wrong_count,beta_u,beta_m,beta_d,no_beta_omega_mean,beta_omega_mean,diagnosis
0,BayesMMRL_16shot_seed1,0.954970,0.956998,0.001217,0.955375,0.956187,0.000406,0.001217,C_only_outputs.logits,0.956998,0,2356,1.0,0.0,0.5,0.005121,0.149808,dynamic fusion beats repo eval; inspect Transi...
1,MMRL_16shot_seed1,0.953753,0.965923,0.008519,0.964300,0.966734,0.010548,0.012982,beta_tuned_logit_dynamic,0.965923,0,2360,0.0,0.0,2.0,0.005267,0.059568,dynamic fusion beats repo eval; inspect Transi...



Accuracy:


,variant,accuracy,n_correct,n,nll,brier,ece_10bin,case_name,split
0,C_only_outputs.logits,0.965923,2381,2465,0.106166,0.051074,0.003124,MMRL_16shot_seed1,test
1,R_only_aux_logits_rep,0.008519,21,2465,7.533094,1.237190,0.400570,MMRL_16shot_seed1,test
2,repo_aux_fusion,0.953753,2351,2465,0.177742,0.074454,0.047139,MMRL_16shot_seed1,test
3,repo_eval_select_eval_logits,0.953753,2351,2465,0.177742,0.074454,0.047139,MMRL_16shot_seed1,test
4,formula_logit_alpha_0.700,0.953753,2351,2465,0.177742,0.074454,0.047139,MMRL_16shot_seed1,test
5,no_beta_logit_dynamic,0.964300,2377,2465,0.109472,0.053431,0.003513,MMRL_16shot_seed1,test
6,beta_tuned_logit_dynamic,0.966734,2383,2465,0.110020,0.052877,0.006402,MMRL_16shot_seed1,test
7,control_prob_prior_0.700C_0.300R,0.965517,2380,2465,0.457563,0.158586,NaN,MMRL_16shot_seed1,test
8,C_only_outputs.logits,0.956998,2359,2465,0.114737,0.058786,0.014613,BayesMMRL_16shot_seed1,test
9,R_only_aux_logits_rep,0.001217,3,2465,7.155358,1.288794,0.460993,BayesMMRL_16shot_seed1,test
